## Setup


In [1]:
import torch
import numpy as np
import pandas as pd
import json
import concurrent.futures
import matplotlib.ticker as mtick
import seaborn as sns
import re
import time
import functools
import ast
import csv
import random

from pathlib import Path
from datasets import Dataset, load_dataset, load_from_disk
from geopy.geocoders import Nominatim
from timezonefinder import TimezoneFinder
from tqdm.notebook import tqdm
from chromadb.utils import embedding_functions
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from transformers import pipeline

from utilities.config import (
    MAX_WORKERS,
    WAIT_TIME,
    RANDOM_SEED,
    BUFFER_SIZE,
    SAMPLE_PERCENTAGE,
    VALIDATION_SAMPLE,
    CHECKPOINT_INTERVAL,
    BATCH_SIZE,
    MODEL_ID,
    OPENAI_API_KEY,
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    WILDCHAT_DATASET,
    CONVERSATION_FIELD,
    LANGUAGE_FIELD,
    TARGET_LANGUAGE,
    PARENT_COLUMN,
    COUNT_VARIABLE,
    MAJOR_CATEGORIES,
    DATA_PATH,
    INPUT_DATA_DIR,
    OUTPUT_DIR,
    LABELED_DATA_DIR,
    PROMPTS_DIR,
    BATCHES_DIR,
    INPUT_BATCHES_DIR,
    OUTPUT_BATCHES_DIR,
    RAW_1P_DATA_PATH,
    RAW_CLAUDE_DATA_PATH,
    TASK_STATEMENTS_PATH,
    TASK_RATING_PATH,
    WILDCHAT_FULL,
    WILDCHAT_ENGLISH,
    WILDCHAT_SAMPLES_FILE,
    TASK_MAPPING_PROMPT_PATH,
    LABOR_TRANSFER_PROMPT_PATH,
    LAST_LEVEL_TASK_MAPPING_PROMPT_PATH,
    TASK_MAPPING_OUTPUT_PATH,
    TIMEZONES_OUTPUT_PATH,
    LABOR_TRANSFER_OUTPUT_FILE,
    LABOR_TRANSFER_BATCH_FILE,
    LABOR_TRANSFER_BATCH_OUTPUT_FILE,
    WORK_RELATED_BATCH_FILE,
    WORK_RELATED_BATCH_OUTPUT_FILE,
    WORK_RELATED_PROMPT_PATH,
    WORK_RELATED_OUTPUT_PATH,
    HIERARCHY_PATH,
    LEVEL_2_BATCH_FILE,
    LEVEL_2_BATCH_OUTPUT_FILE,
    LEVEL_1_BATCH_FILE,
    LEVEL_1_BATCH_OUTPUT_FILE,
    LEVEL_0_BATCH_FILE,
    LEVEL_0_BATCH_OUTPUT_FILE,
)
from utilities.batch_utils import (
    _create_batch_file,
    _submit_batch_file,
    _retrieve_batch_job_results,
    submit_and_retrieve,
)
from utilities.llm_utils import (
    RobustEncoder,
    get_messages,
    build_messages,
    strip_messages,
    format_conversation,
    get_gpt_response,
    parallelize_llm_call,
    format_tasks,
    format_options,
    format_last_level_options,
)

# from clio.hierarchizer import

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [2]:
client = OpenAI(api_key=OPENAI_API_KEY)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# Functions for batch requests
submit_batch_file = functools.partial(_submit_batch_file, client=client)
retrieve_batch_job_results = functools.partial(
    _retrieve_batch_job_results, client=client
)
submit_and_retrieve = functools.partial(submit_and_retrieve, client=client)

# Functions for direct API calls
get_gpt_response = functools.partial(get_gpt_response, client=client)
parallelize_llm_call = functools.partial(parallelize_llm_call, client=client)

In [4]:
if not WILDCHAT_FULL.exists():
    wildchat_ds = load_dataset(
        path=WILDCHAT_DATASET,
        split="train",
    )
    wildchat_ds.save_to_disk(WILDCHAT_FULL)

if not WILDCHAT_ENGLISH.exists():
    wildchat_ds = load_from_disk(WILDCHAT_FULL)
    english_conversations_rows = wildchat_ds.filter(
        lambda x: x[LANGUAGE_FIELD] == TARGET_LANGUAGE
    )
    english_conversations_rows.save_to_disk(WILDCHAT_ENGLISH)
else:
    english_conversations_rows = load_from_disk(WILDCHAT_ENGLISH)

total_rows = len(english_conversations_rows)
sample_size = int(SAMPLE_PERCENTAGE * total_rows)
sample_conversations_rows = (
    english_conversations_rows.shuffle(
        seed=RANDOM_SEED,
    )
    .select(range(sample_size))
    .to_pandas()
)
print(f"Total conversations: {total_rows}, sample size: {sample_size}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total conversations: 1679371, sample size: 58777


In [5]:
sample_conversations_rows.shape

(58777, 14)

In [6]:
sample_conversations_rows.columns.values

array(['conversation_hash', 'model', 'timestamp', 'conversation', 'turn',
       'language', 'openai_moderation', 'detoxify_moderation', 'toxic',
       'redacted', 'state', 'country', 'hashed_ip', 'header'],
      dtype=object)

In [7]:
wildchat_columns_to_keep = [CONVERSATION_FIELD, "timestamp", "country", "state"]
sample_conversations = sample_conversations_rows[wildchat_columns_to_keep].copy()
sample_conversations.shape

(58777, 4)

In [8]:
sample_conversations["conversation"] = sample_conversations["conversation"].apply(
    lambda x: [dict(msg) for msg in x] if not isinstance(x, list) else x
)

sample_conversations["conversation"] = sample_conversations["conversation"].apply(
    lambda conv: strip_messages(conv) if isinstance(conv, list) else None
)
sample_conversations.drop_duplicates(subset=["conversation"], inplace=True)
sample_conversations.shape

(52489, 4)

In [3]:
device = ""

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [4]:
client = OpenAI(
    api_key=OPENAI_API_KEY,
)

## Conversation-task mapping

In order to map conversations to tasks, we are using the following steps:

1. Retrieve the conversations from Wildchat, sampling for English conversations
2. Filter out all the non-work-related conversations
3. Map the conversations to tasks using a prompt that asks the model to identify the task associated with each conversation.


### Work-related conversations filtering


In [5]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df.columns

Index(['O*NET-SOC Code', 'Title', 'Task ID', 'Task'], dtype='object')

In [6]:
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)

In [13]:
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,"Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency.",Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assign or delegate responsibilities to them.,Management Occupations
2,11-1011.00,Chief Executives,8825,"Analyze operations to evaluate performance of a company or its staff in meeting objectives or to determine areas of potential cost reduction, program improvement, or policy change.",Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objectives, or activities of organizations or businesses to ensure continuing operations, to maximize returns on investments, or to increase productivity.",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those for funding or implementation of programs.",Management Occupations


In [14]:
def parse_work_related_batch_results(output_path: Path):
    results = {}
    with open(output_path, "r") as f:
        for line in f:
            record = json.loads(line)

            if record.get("error") is not None:
                results[record["custom_id"]] = "ERROR"
                continue

            custom_id = record["custom_id"]
            content = record["response"]["body"]["choices"][0]["message"]["content"]

            try:
                parsed_content = json.loads(content)
                if "answer" in parsed_content:
                    val = parsed_content["answer"]
                    if val in (1, "1"):
                        answer = "Yes"
                    elif val in (0, "0"):
                        answer = "No"
            except json.JSONDecodeError:
                answer = "Unknown"

            if answer == "Unknown":
                print(
                    f"Unexpected response for {custom_id}. Complete content: {content}"
                )

            results[custom_id] = answer

    return results


def filter_work_conversations(conversations: pd.DataFrame, path: Path) -> list:
    """
    Filter conversations to determine which ones are work-related using the GPT model.

    :param conversations: DataFrame containing conversations to filter
    :param path: Path to save the filtered conversations
    :return: list containing boolean values indicating whether each conversation is work-related or not
    """
    formatted_messages = []
    template = get_messages(path=WORK_RELATED_PROMPT_PATH)
    word_id = path.stem.split("_")[0]

    for index, row in conversations.iterrows():
        raw_conversation = row["conversation"]
        raw_conversation = strip_messages(raw_conversation)
        conversation = format_conversation(conversation=raw_conversation)
        messages = build_messages(template=template, conversation=conversation)
        formatted_messages.append(messages)

    WORK_RELATED_BATCH_FILE_750 = INPUT_BATCHES_DIR / f"{word_id}_750.jsonl"
    WORK_RELATED_BATCH_OUTPUT_FILE_750 = OUTPUT_BATCHES_DIR / f"{word_id}_750.jsonl"

    results = submit_and_retrieve(
        messages=formatted_messages,
        batch_file=WORK_RELATED_BATCH_FILE_750,  # WORK_RELATED_BATCH_FILE,
        output_file=WORK_RELATED_BATCH_OUTPUT_FILE_750,  # WORK_RELATED_BATCH_OUTPUT_FILE,
        word_id=word_id,
        parse_func=parse_work_related_batch_results,
    )
    answers = [results.get(f"{word_id}_{i}") for i in range(len(formatted_messages))]
    return answers

In [ ]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        conversations=sample_conversations, path=WORK_RELATED_OUTPUT_PATH
    )
    sample_conversations["is_work_related_model"] = answers
    sample_conversations["timestamp"] = sample_conversations["timestamp"].apply(
        lambda x: x.isoformat() if hasattr(x, "isoformat") else str(x)
    )
    sample_conversations.to_csv(WORK_RELATED_OUTPUT_PATH, index=False)
    work_related_conversations_df = sample_conversations[
        sample_conversations["is_work_related_model"] == "Yes"
    ].copy()
else:
    work_related_conversations_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)
work_related_conversations_df.shape

Submitting messages for mini_750: total 750 messages in 1 batches of up to 5000 messages each
Skipping batch 0 (750 requests): data/batches/output/mini_750_0.jsonl already exists
[0] batches already exist. Skipping those batches and retrieving results from existing files.


(750, 3)

In [15]:
def get_timezone_for_location(
    state: str, country: str, geolocator: Nominatim, tf: TimezoneFinder
) -> list[str]:
    try:
        query = f"{state}, {country}" if pd.notnull(state) else country
        location = geolocator.geocode(query)
        print(f"Geocoding query: '{query}' -> Location: {location}", flush=True)
        if location:
            print(
                f"Found location for query '{query}': {location.address} (lat: {location.latitude}, lng: {location.longitude})",
                flush=True,
            )
            return tf.timezone_at(lng=location.longitude, lat=location.latitude)
    except Exception as e:
        print(
            f"Error geocoding location for state '{state}' and country '{country}': {e}",
            flush=True,
        )


def find_timezones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find timezones for locations in the DataFrame.

    :param df: Input DataFrame containing location information.
    :return: DataFrame with an additional column 'timezone' containing the found timezones.
    """

    unique_locations = df[["state", "country"]].drop_duplicates()
    timezones_df = pd.DataFrame(columns=["state", "country", "timezone"])
    tf = TimezoneFinder(in_memory=True)
    geolocator = Nominatim(user_agent="chatgpt_at_work_project")

    for _, row in unique_locations.iterrows():
        state = row["state"]
        country = row["country"]
        timezone = get_timezone_for_location(
            state=state, country=country, geolocator=geolocator, tf=tf
        )
        time.sleep(1)  # Sleep to respect geocoding service rate limits
        record = {"state": state, "country": country, "timezone": timezone}
        timezones_df = pd.concat(
            [timezones_df, pd.DataFrame([record])], ignore_index=True
        )

    df = df.merge(timezones_df, on=["state", "country"], how="left")
    return df

In [ ]:
if not TIMEZONES_OUTPUT_PATH.exists():
    work_related_conversations_df = find_timezones(df=work_related_conversations_df)
    work_related_conversations_df.to_csv(TIMEZONES_OUTPUT_PATH, index=False)
else:
    work_related_conversations_df = pd.read_csv(TIMEZONES_OUTPUT_PATH)
work_related_conversations_df.shape

(36680, 6)

In [ ]:
work_related_conversations_df = work_related_conversations_df[
    work_related_conversations_df["timezone"].notnull()
]
work_related_conversations_df.shape

(36131, 6)

In [ ]:
def normalize_timezone(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize timezone by converting UTC to the local timezone.

    :param df: Input DataFrame containing a 'timezone' column to normalize.
    :return: DataFrame with an additional column 'normalized_timezone' containing the normalized timezone names.
    """
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize("UTC")

    df["timestamp_local"] = df.apply(
        lambda row: row["timestamp"].tz_convert(row["timezone"]), axis=1
    )
    return df

In [38]:
work_related_conversations_df = normalize_timezone(df=work_related_conversations_df)
work_related_conversations_df.shape

(36131, 7)

In [260]:
work_related_conversations_df.columns

Index(['conversation', 'timestamp', 'is_work_related', 'country', 'state',
       'timezone', 'timestamp_local'],
      dtype='object')

### Task mapping


In [ ]:
hierarchy = json.load(open(HIERARCHY_PATH, "r"))
professions = tasks_df["Title"].tolist()
hierarchy["levels"][0]["professions"] = professions


In [ ]:
# Trova tutti gli indici associati ad 'Animal Scientists' e stampa i rispettivi task (items) assegnati nel livello 0
animal_scientists_indices = [
    i
    for i, prof in enumerate(hierarchy["levels"][0]["professions"])
    if prof == "Animal Scientists"
]

print(f"Trovati {len(animal_scientists_indices)} task per Animal Scientists:")
for i in animal_scientists_indices:
    print("-", hierarchy["levels"][0]["items"][i])

Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Executives
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Officers
Chief Sustainability Of

In [55]:
def parse_task_mapping_batch_results(
    output_path: Path, n_batches: int
) -> dict[str, str | list[str]]:
    results = {}

    for i in range(n_batches):
        current_path = output_path.with_stem(f"{output_path.stem}_{i}")
        if not current_path.exists():
            continue
        with open(current_path, "r") as f:
            for line in f:
                record = json.loads(line)

                if record.get("error") is not None:
                    results[record["custom_id"]] = "ERROR"
                    continue

                custom_id = record["custom_id"]
                content = record["response"]["body"]["choices"][0]["message"]["content"]

                answer = "Unknown"
                try:
                    parsed_content = json.loads(content)

                    if "answer" in parsed_content:
                        answer = parsed_content["answer"]
                    elif (
                        "command" in parsed_content
                        and "args" in parsed_content["command"]
                    ):
                        content = parsed_content["command"]["args"]
                        if isinstance(content, dict):
                            answer = content.get("message", "Unknown")
                        else:
                            answer = "Unknown"
                except (KeyError, ValueError, json.JSONDecodeError):
                    print(
                        f"Wrong JSON format for {custom_id}, content: {content[:100]}"
                    )

                results[custom_id] = answer

    return results

In [56]:
# TODO: Pass batch file and output file as parameters to avoid hardcoding


def map_conversation_to_task(
    conversations: pd.DataFrame, tasks: dict, path: Path
) -> pd.DataFrame:
    """
    Maps user conversations to hierarchical tasks at multiple levels (levels 2, 1, and 0). The function takes in a
    DataFrame of conversations, a DataFrame of tasks containing hierarchical task definitions, and a file path where
    the resulting mappings will be saved. It processes tasks hierarchy starting from level 2 (high-level) tasks,
    then moves to level 1 (medium-level), and finally to level 0 (low-level) tasks. For each conversation, tasks
    are determined progressively based on parent-child relationships within the tasks DataFrame.

    :param conversations: A pandas DataFrame containing conversations. Each row should represent a conversation.
    :param tasks: A pandas DataFrame containing hierarchical task definitions. Each task should include information
                  about its level (0, 1, 2) and its parent task association if applicable.
    :param path: A Path object specifying the file path to save the mapping of conversations to tasks.
    :return: A pandas DataFrame containing the original conversations and their corresponding mapped hierarchical
             tasks at levels 2, 1, and 0.
    """
    template = get_messages(path=TASK_MAPPING_PROMPT_PATH)
    last_level_template = get_messages(path=LAST_LEVEL_TASK_MAPPING_PROMPT_PATH)
    conversations_list = conversations["conversation"].tolist()
    n_batches = len(conversations_list) // BATCH_SIZE + 1

    # Level 2
    high_level_tasks = tasks["levels"][2]["items"]
    options_str = format_options(tasks=high_level_tasks)
    messages = [
        build_messages(
            template=template,
            conversation=format_conversation(conv),
            options_str=options_str,
        )
        for conv in conversations_list
    ]
    results = submit_and_retrieve(
        messages=messages,
        batch_file=LEVEL_2_BATCH_FILE,
        output_file=LEVEL_2_BATCH_OUTPUT_FILE,
        word_id="level2",
        parse_func=lambda _: parse_task_mapping_batch_results(
            output_path=LEVEL_2_BATCH_OUTPUT_FILE, n_batches=n_batches
        ),
    )
    level_2_responses = [
        results.get(f"level2_{i}").strip() if results.get(f"level2_{i}") else "Unknown"
        for i in range(len(conversations_list))
    ]

    # Level 1
    level_1_messages = []
    for conv, level_2_response in zip(conversations_list, level_2_responses):
        parent_id = high_level_tasks.index(level_2_response)
        medium_level_tasks = tasks["levels"][1]["items"]
        medium_level_parent_indices = tasks["levels"][1]["parent_indices"]
        filtered_medium_tasks = [
            task
            for task, parent_idx in zip(medium_level_tasks, medium_level_parent_indices)
            if parent_idx == parent_id
        ]
        options_str = format_options(tasks=filtered_medium_tasks)
        level_1_messages.append(
            build_messages(
                template=template,
                conversation=format_conversation(conv),
                options_str=options_str,
            )
        )
    results = submit_and_retrieve(
        messages=level_1_messages,
        batch_file=LEVEL_1_BATCH_FILE,
        output_file=LEVEL_1_BATCH_OUTPUT_FILE,
        word_id="level1",
        parse_func=lambda _: parse_task_mapping_batch_results(
            output_path=LEVEL_1_BATCH_OUTPUT_FILE, n_batches=n_batches
        ),
    )
    level_1_responses = [
        results.get(f"level1_{i}").strip() if results.get(f"level1_{i}") else "Unknown"
        for i in range(len(conversations_list))
    ]

    # Level 0
    level_0_messages = []
    for conv, level_1_response in zip(conversations_list, level_1_responses):
        parent_id = medium_level_tasks.index(level_1_response)
        low_level_tasks = tasks["levels"][0]["items"]
        low_level_parent_indices = tasks["levels"][0]["parent_indices"]
        filtered_low_level_tasks = [
            task
            for task, parent_idx in zip(low_level_tasks, low_level_parent_indices)
            if parent_idx == parent_id
        ]
        low_level_professions = tasks["levels"][0]["professions"]
        low_level_professions = [
            profession
            for profession, parent_idx in zip(
                low_level_professions, low_level_parent_indices
            )
            if parent_idx == parent_id
        ]
        options_str = format_last_level_options(
            professions=low_level_professions, tasks=filtered_low_level_tasks
        )
        level_0_messages.append(
            build_messages(
                template=last_level_template,
                conversation=format_conversation(conv),
                options_str=options_str,
            )
        )
    results = submit_and_retrieve(
        messages=level_0_messages,
        batch_file=LEVEL_0_BATCH_FILE,
        output_file=LEVEL_0_BATCH_OUTPUT_FILE,
        word_id="level0",
        parse_func=lambda _: parse_task_mapping_batch_results(
            output_path=LEVEL_0_BATCH_OUTPUT_FILE, n_batches=n_batches
        ),
    )
    level_0_responses = [
        results.get(f"level0_{i}") if results.get(f"level0_{i}") else "Unknown"
        for i in range(len(conversations_list))
    ]

    result = pd.DataFrame(
        {
            "conversation": conversations_list,
            "level_2_task": level_2_responses,
            "level_1_task": level_1_responses,
            "level_0_task": level_0_responses,
        }
    )
    result.to_csv(path, index=False)
    return result

In [ ]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    task_mapped_conversations_df = map_conversation_to_task(
        conversations=work_related_conversations_df,
        tasks=tasks_df,
        path=TASK_MAPPING_OUTPUT_PATH,
    )
else:
    task_mapped_conversations_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)
task_mapped_conversations_df.shape

(50381, 4)

In [ ]:
def check_consensus(items: list[str]) -> str | None:
    """
    Check if there is a consensus of profession at level 0.

    :param items: List of professions for a given conversation.
    :return: The profession:task if there is a consensus, None otherwise.
    """
    profession_stats = {}
    print(f"Checking consensus for items: {items}")
    try:
        items = ast.literal_eval(items)
        for item in items:
            print(f"Processing item: {item}")
            profession, task = item.split(":")
            profession_stats[profession] = {
                "count": profession_stats.get(profession, {"count": 0})["count"] + 1,
                "task": task,
            }

        max_count = max(
            profession_stats[profession]["count"] for profession in profession_stats
        )
        if max_count >= len(items) / 2:
            for profession, stats in profession_stats.items():
                if stats["count"] == max_count:
                    return f"{profession}:{stats['task']}"
        else:
            return None
    except Exception:
        print(f"Error processing items: {items}")
        return None


def filter_task_mappings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter task mappings to keep only those where there is a consensus of profession at level 0.
    This means that for a given conversation, the level 0 should contain a majority profession among the options provided.
    At the end, only the profession:task pair that has a majority profession will be kept for each conversation.

    :param df: Input DataFrame containing task mappings with columns 'level_2_task', 'level_1_task', and 'level_0_task'.
    :return: Filtered DataFrame containing only rows where there is a consensus of profession at level 0
    """
    filtered_df = df.copy()
    filtered_df["level_0_task"] = filtered_df["level_0_task"].apply(
        lambda x: check_consensus(x)
    )
    filtered_df = filtered_df.dropna(subset=["level_0_task"])
    return filtered_df

In [33]:
task_mapped_conversations_df = filter_task_mappings(df=task_mapped_conversations_df)
task_mapped_conversations_df.shape

NameError: name 'task_mapped_conversations_df' is not defined

In [ ]:
task_mapped_conversations_df["job_title"] = task_mapped_conversations_df[
    "level_0_task"
].apply(lambda x: x.split(":")[0] if pd.notnull(x) else None)

task_mapped_conversations_df["level_0_task"] = task_mapped_conversations_df[
    "level_0_task"
].apply(lambda x: x.split(":")[1] if pd.notnull(x) else None)
task_mapped_conversations_df.shape

In [ ]:
task_mapped_conversations_df = format_tasks(
    df=task_mapped_conversations_df, column_name="level_0_task"
)

In [ ]:
test = pd.merge(
    task_mapped_conversations_df,
    work_related_conversations_df,
    on="conversation",
    how="inner",
)

## Labor transfer

In this section, we analyze how ChatGPT transfers work from professionals to users.


In [ ]:
tasks_grouped = (
    tasks_df.groupby("task")["title"]
    .apply(lambda x: ", ".join(x.unique()))
    .reset_index()
)

roles_titles_df = roles_df.merge(
    tasks_grouped, left_on="level_0_task", right_on="task", how="left"
)
roles_titles_df.drop(columns=["task"], inplace=True)
roles_titles_df.shape

(18037, 19)

In [352]:
sample_50 = roles_titles_df.sample(50, random_state=RANDOM_SEED)
conversations_sample_50 = sample_50[["conversation", "level_0_task", "title"]]
conversations_sample_50.to_csv("conversations_sample_50.csv", index=False)

In [ ]:
def parse_labor_transfer_batch_results(output_path: Path) -> dict:
    results = {}
    with open(output_path, "r") as f:
        for line in f:
            record = json.loads(line)
            custom_id = record["custom_id"]

            if record.get("error") is not None:
                results[custom_id] = None
                continue

            content = record["response"]["body"]["choices"][0]["message"]["content"]

            try:
                parsed_content = json.loads(content)
                results[custom_id] = parsed_content
            except (json.JSONDecodeError, KeyError, TypeError) as e:
                print(f"Error parsing {custom_id}: {e}")
                results[custom_id] = None

            if "LT0" in content:
                results[custom_id] = "LT0"
            elif "LT1" in content:
                results[custom_id] = "LT1"
            elif "LT2" in content:
                results[custom_id] = "LT2"

    return results


def analyze_labor_transfer(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyzes the potential labor transfer between two specified roles by comparing the conversations associated with each role.
    The function identifies conversations that are similar between the two roles based on their content and calculates a similarity score.

    :param df: DataFrame containing the conversations and their associated roles
    :param source_role: The name of the role from which labor transfer is being analyzed
    :param target_role: The name of the role to which labor transfer is being analyzed
    :return: A DataFrame containing pairs of conversations from the source and target roles along with their similarity scores
    """
    template = get_messages(path=LABOR_TRANSFER_PROMPT_PATH)
    formatted_messages = []
    word_id = "labor_transfer"

    for idx, row in df.iterrows():
        conversation = row["conversation"]
        title = row["title"]
        task = row["level_0_task"]
        messages = build_messages(
            template=template, conversation=conversation, title=title, task=task
        )
        formatted_messages.append(messages)

    results = submit_and_retrieve(
        messages=formatted_messages,
        batch_file=LABOR_TRANSFER_BATCH_FILE,
        output_file=LABOR_TRANSFER_BATCH_OUTPUT_FILE,
        word_id=word_id,
        parse_func=parse_labor_transfer_batch_results,
    )

    labels = [
        results.get(f"{word_id}_{i}", "Unknown").strip()
        for i in range(len(formatted_messages))
    ]

    return labels

In [355]:
if not LABOR_TRANSFER_OUTPUT_FILE.exists():
    labor_transfer_labels = analyze_labor_transfer(
        df=sample_50,
    )
    labor_transfer_df = pd.DataFrame({"label": labor_transfer_labels})
    labor_transfer_df.to_csv(LABOR_TRANSFER_OUTPUT_FILE, index=False)
else:
    labor_transfer_df = pd.read_csv(LABOR_TRANSFER_OUTPUT_FILE)
labor_transfer_df.shape

Submitting messages for labor_transfer_0.03_output: total 50 messages in 1 batches of up to 5000 messages each
[] batches already exist. Skipping those batches and retrieving results from existing files.
Submitted batch 0 (50 requests): batch_6a0719cd9a788190acd9e452273b5a4f
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status validating. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status validating. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status in_progress. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status in_progress. Waiting for 60 seconds before retrying...


(50, 1)

In [366]:
roles_df.columns

Index(['conversation', 'timestamp', 'is_work_related', 'level_2_task',
       'level_1_task', 'level_0_task', 'Function', 'orig_index', 'interaction',
       'social_dimensions', 'role_sentence', 'role', 'role_core',
       'embedding_role_core', 'cluster_id', 'emotion', 'timestamp_local',
       'hour'],
      dtype='object')

In [ ]:
test = roles_df[roles_df["role"] == "Code reviewer"]

In [ ]:
batch_output_file = "data/batches/output/labor_transfer_0.03_output_0.jsonl"
csv_sample_path = "conversations_sample_50.csv"

parsed_results = []

with open(batch_output_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        try:
            data = json.loads(line)
            custom_id = data.get("custom_id", "")

            row_idx = int(custom_id.split("_")[-1])

            choices = data.get("response", {}).get("body", {}).get("choices", [])
            if not choices:
                continue

            content_str = choices[0].get("message", {}).get("content", "{}")
            content_dict = json.loads(content_str)

            parsed_results.append(
                {
                    "original_index": row_idx,
                    "label": content_dict.get("label"),
                    "lt1_reason": content_dict.get("lt1_reason"),
                    "task_match": content_dict.get("task_match"),
                }
            )

        except Exception as e:
            print(f"Errore nel parsing della riga con custom_id '{custom_id}': {e}")

df_parsed = pd.DataFrame(parsed_results)

df_sample_50 = pd.read_csv(csv_sample_path)

df_final = df_sample_50.merge(
    df_parsed, left_index=True, right_on="original_index", how="left"
)

df_final = df_final.drop(columns=["original_index"]).reset_index(drop=True)

df_final.to_csv("labor_transfer_analysis.csv", index=False)

display(df_final.head(1))

conversation  \
0  [{'role': 'user', 'content': 'que opinas de este codigo y calidad?\nimport asyncio\nimport aiohttp\nfrom bs4 import BeautifulSoup\nimport json\nfrom typing import List, Dict, Optional\nimport ssl\nimport logging\nfrom datetime import datetime\nimport colorlog\n\n# Configure logging\nhandler = colorlog.StreamHandler()\nhandler.setFormatter(colorlog.ColoredFormatter(\n    \'%(log_color)s%(asctime)s [%(levelname)s] %(message)s\',\n    datefmt=\'%Y-%m-%d %H:%M:%S\',\n    log_colors={\n        \'DEBUG\':    \'light_cyan\',\n        \'INFO\':     \'light_green\',\n        \'WARNING\':  \'light_yellow\',\n        \'ERROR\':    \'light_red\',\n        \'CRITICAL\': \'light_red,bg_white\',\n    }\n))\n\nlogger = colorlog.getLogger(__name__)\nlogger.addHandler(handler)\nlogger.setLevel(logging.INFO)\n\n# Remove any existing handlers to avoid duplicate logs\nfor hdlr in logger.handlers[:-1]:\n    logger.removeHandler(hdlr)\n\n# Configuration settings\nCONFIG = {\n    \'base_url\': \'https://10.0.0.2\',\n    \'username\': \'misael_campos\',\n    \'password\': \'d337O8AetR3p4\',\n    \'output_json\': \'aggregated_pon_onu_data.json\',\n    \'post_endpoint\': \'/action/pononuopticalinfo.html\',\n    \'who\': 100,\n    \'onuid\': 0,\n    \'pon_range\': range(1, 17),\n    \'onu_group_range\': range(0, 2),\n    \'request_timeout\': aiohttp.ClientTimeout(\n        total=240,      # Total operation timeout\n        connect=120,   # Connection timeout\n        sock_read=120  # Socket read timeout\n    ),\n    \'max_concurrent_requests\': 8,  # Reduced to prevent server overload\n    \'max_retries\': 3,              # Maximum number of retry attempts\n    \'retry_delay\': 1,              # Delay between retries in seconds\n    \'batch_size\': 8                # Number of concurrent requests per batch\n}\n\nclass AsyncScraper:\n    def __init__(self, base_url: str):\n        self.base_url = base_url.rstrip(\'/\')\n        self.ssl_context = ssl.create_default_context()\n        self.ssl_context.check_hostname = False\n        self.ssl_context.verify_mode = ssl.CERT_NONE\n        self.session = None\n        self.sem = asyncio.Semaphore(CONFIG[\'max_concurrent_requests\'])\n\n    async def __aenter__(self):\n        timeout = CONFIG[\'request_timeout\']\n        self.session = aiohttp.ClientSession(\n            connector=aiohttp.TCPConnector(\n                ssl=self.ssl_context,\n                limit=CONFIG[\'max_concurrent_requests\']\n            ),\n            timeout=timeout,\n            headers={\'User-Agent\': \'Mozilla/5.0 (Windows NT 10.0; Win64; x64)\'}\n        )\n        return self\n\n    async def __aexit__(self, exc_type, exc_val, exc_tb):\n        if self.session:\n            await self.session.close()\n\n    async def login(self, username: str, password: str) -> bool:\n        login_url = f"{self.base_url}/action/main.html"\n        login_data = {\n            \'user\': username,\n            \'pass\': password,\n            \'button\': \'Login\',\n            \'who\': \'100\',\n        }\n        \n        for attempt in range(CONFIG[\'max_retries\']):\n            try:\n                async with self.session.post(login_url, data=login_data) as response:\n                    text = await response.text()\n                    if "Sorry, you do not have access" in text:\n                        logger.error("Login failed: Access denied")\n                        return False\n                    elif "Error" in text or "Incorrect password" in text:\n                        logger.error("Login failed: Incorrect credentials")\n                        return False\n                    logger.info("Login successful!")\n                    return True\n            except Exception as e:\n                logger.error(f"Login attempt {attempt + 1} failed: {e}")\n                if attempt < CONFIG[\'max_retries\'] - 1:\n                    await asyncio.sleep(CONFIG[\'retry_delay\'])\n                else:\n         

In [358]:
sample_50["labor_transfer_label"] = labor_transfer_labels
sample_50.columns

Index(['conversation', 'timestamp', 'is_work_related', 'level_2_task',
       'level_1_task', 'level_0_task', 'Function', 'orig_index', 'interaction',
       'social_dimensions', 'role_sentence', 'role', 'role_core',
       'embedding_role_core', 'cluster_id', 'emotion', 'timestamp_local',
       'hour', 'title', 'labor_transfer_label'],
      dtype='object')

In [359]:
temp = sample_50[["conversation", "level_0_task", "title", "labor_transfer_label"]]
temp.to_csv("conversations_sample_50_with_labels.csv", index=False)

## Validation

This is gonna contain the validation of task-conversation mapping, Automation/augmentation labels and role extraction.


### CLIO Hierarchy validation


In [15]:
HIERARCHY_TASK_VALIDATION = OUTPUT_DIR / "hierarchy_task_validation.csv"

macro_level = hierarchy["levels"][2]["items"]
mid_level = hierarchy["levels"][1]["items"]
leaf_level = hierarchy["levels"][0]["items"]

mid_to_macro = hierarchy["levels"][1]["parent_indices"]
leaf_to_mid = hierarchy["levels"][0]["parent_indices"]

sample_indices = random.sample(range(len(leaf_level)), 150)

if not HIERARCHY_TASK_VALIDATION.exists():
    with open(HIERARCHY_TASK_VALIDATION, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Task_L0", "Cluster_L1", "Cluster_L2", "Agreement"])

        for idx in sample_indices:
            task_l2 = leaf_level[idx]
            mid_idx = leaf_to_mid[idx]

            if mid_idx is not None and mid_idx < len(mid_level):
                task_l1 = mid_level[mid_idx]
                macro_idx = mid_to_macro[mid_idx]

                if macro_idx is not None and macro_idx < len(macro_level):
                    task_l0 = macro_level[macro_idx]
                    writer.writerow([task_l2, task_l1, task_l0, ""])
                else:
                    writer.writerow([task_l2, task_l1, "Error: L2 not found", ""])
            else:
                writer.writerow([task_l2, "Error: L1 not found", "N/A", ""])
    print(f"File '{HIERARCHY_TASK_VALIDATION}' was successfully generated.")
else:
    print(f"File '{HIERARCHY_TASK_VALIDATION}' already exists. Skipping generation.")


File 'data/output/hierarchy_task_validation.csv' already exists. Skipping generation.


In [17]:
hierarchy_agreement = pd.read_csv(HIERARCHY_TASK_VALIDATION)
total_elements = len(hierarchy_agreement)
agreement_counts = len(hierarchy_agreement[hierarchy_agreement["Agreement"] == "Yes"])
agreement_percentage = (agreement_counts / total_elements) * 100
print(f"Agreement percentage: {agreement_percentage:.2f}%")

Agreement percentage: 97.33%


### Work-related conversation validation


In [11]:
LABELED_SAMPLE_PATH = LABELED_DATA_DIR / "work_related_750.csv"
WORK_RELATED_OUTPUT_750 = OUTPUT_DIR / "mini_work_related_750.csv"

sample_750_df = pd.read_csv(LABELED_SAMPLE_PATH)
sample_750_df["conversation"] = sample_750_df["conversation"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
sample_750_df.shape

(750, 2)

In [57]:
if not WORK_RELATED_OUTPUT_750.exists():
    answers = filter_work_conversations(
        conversations=sample_750_df,
        path=WORK_RELATED_OUTPUT_750,
    )
    sample_750_df["is_work_related_model"] = answers
    sample_750_df.to_csv(WORK_RELATED_OUTPUT_750, index=False)
else:
    sample_750_df = pd.read_csv(WORK_RELATED_OUTPUT_750)
sample_750_df.shape

(750, 3)

In [58]:
sample_750_df.columns

Index(['conversation', 'is_work_related_human', 'is_work_related_model'], dtype='object')

In [59]:
cohen = cohen_kappa_score(
    sample_750_df["is_work_related_human"],
    sample_750_df["is_work_related_model"],
)
print(f"Cohen's kappa: {cohen}")

Cohen's kappa: 0.6551609984330515


In [15]:
y_pred = sample_750_df["is_work_related_model"].values
y_true = sample_750_df["is_work_related_human"].values

tp = sum((y_pred == "Yes") & (y_true == "Yes"))
tn = sum((y_pred == "No") & (y_true == "No"))
fp = sum((y_pred == "Yes") & (y_true == "No"))
fn = sum((y_pred == "No") & (y_true == "Yes"))

total_valid = len(sample_750_df)
accuracy = (tp + tn) / total_valid if total_valid > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0


print(f"Accuracy: {accuracy:.2%}")
print(f"False Positive Rate: {fpr:.2%}")
print(f"True Positive Rate: {tpr:.2%}\n\n")

print(pd.crosstab(y_true, y_pred, rownames=["Actual"], colnames=["Predicted"]))

Accuracy: 83.33%
False Positive Rate: 14.84%
True Positive Rate: 82.23%


Predicted   No  Yes
Actual             
No         241   42
Yes         83  384


In [60]:
sample_750_df = sample_750_df[sample_750_df["is_work_related_model"] == "Yes"]
sample_750_df.shape

(426, 3)

### Task-conversation mapping validation


In [61]:
TASK_MAPPING_750 = OUTPUT_DIR / "task_mapping_750.csv"

In [65]:
temp_150_sample_df = sample_750_df.sample(150, random_state=RANDOM_SEED)
temp_150_sample_df.shape

(150, 3)

In [79]:
if not TASK_MAPPING_750.exists():
    validation_task_mapping = map_conversation_to_task(
        conversations=temp_150_sample_df[["conversation"]],
        tasks=hierarchy,
        path=TASK_MAPPING_750,
    )

    validated_work_related_only = pd.merge(
        left=temp_150_sample_df,
        right=validation_task_mapping,
        left_on="conversation",
        right_on="conversation",
        how="inner",
    )

    validated_work_related_only.to_csv(TASK_MAPPING_750, index=False)
else:
    validated_work_related_only = pd.read_csv(TASK_MAPPING_750)
validated_work_related_only.shape

(150, 6)

In [80]:
validated_work_related_only = filter_task_mappings(df=validated_work_related_only)
validated_work_related_only.shape

Checking consensus for items: ["Computer and Information Systems Managers: Provide users with technical support for computer problems.",
 "Computer User Support Specialists: Answer user inquiries regarding computer software or hardware operation to resolve problems.",
 "Web Administrators: Provide training or technical assistance in web site implementation or use.",
 "Document Management Specialists: Consult with end users regarding problems in accessing electronic content.",
 "Office Clerks, General: Troubleshoot problems involving office equipment, such as computer hardware and software."]
Processing item: Computer and Information Systems Managers: Provide users with technical support for computer problems.
Processing item: Computer User Support Specialists: Answer user inquiries regarding computer software or hardware operation to resolve problems.
Processing item: Web Administrators: Provide training or technical assistance in web site implementation or use.
Processing item: Docume

(12, 6)

In [81]:
validated_work_related_only["job_title"] = validated_work_related_only[
    "level_0_task"
].apply(lambda x: x.split(":")[0] if pd.notnull(x) else None)

validated_work_related_only["level_0_task"] = validated_work_related_only[
    "level_0_task"
].apply(lambda x: x.split(":")[1] if pd.notnull(x) else None)
validated_work_related_only.shape

(12, 7)

In [82]:
validated_work_related_only.shape

(12, 7)

In [83]:
CONSENSUS_TASK_MAPPING_750 = OUTPUT_DIR / "consensus_task_mapping_750.csv"
validated_work_related_only.to_csv(CONSENSUS_TASK_MAPPING_750, index=False)

In [ ]:
task_agreement = len(
    validated_work_related_only[validated_work_related_only["task_plausible"] == "Yes"]
) / len(validated_work_related_only)
task_agreement

0.8615384615384616

### Labor transfer validation


In [ ]:
LABOR_TRANSFER_OUTPUT_750 = OUTPUT_DIR / "labor_transfer_750.csv"

sample_50_df = temp_150_sample_df.sample(
    min(50, temp_150_sample_df.shape[0]), random_state=RANDOM_SEED
)

In [ ]:
if not LABOR_TRANSFER_OUTPUT_750.exists():
    labor_transfer_jsons = analyze_labor_transfer(
        df=sample_50,
    )
    labor_transfer_df = pd.DataFrame({"labor_transfer": labor_transfer_jsons})
    labor_transfer_df.to_csv(LABOR_TRANSFER_OUTPUT_750, index=False)
else:
    labor_transfer_df = pd.read_csv(LABOR_TRANSFER_OUTPUT_750)
labor_transfer_df.shape

In [ ]:
sample_50_df["labor_transfer"] = labor_transfer_jsons
sample_50_df.columns

In [ ]:
rows = pd.DataFrame(
    "interaction_type",
    "task_match",
    "label",
    "lt1_reason",
    "transferred_from",
    "transferred_from_other",
    "rationale",
    "confidence",
)
for idx, row in sample_50_df.iterrows():
    json = row["labor_transfer"]
    conversation_data = ast.literal_eval(json)

    interaction_type = conversation_data.get("interaction_type")
    task_match = conversation_data.get("task_match")
    label = conversation_data.get("label")
    lt1_reason = conversation_data.get("lt1_reason")
    transferred_from = conversation_data.get("transferred_from")
    transferred_from_other = conversation_data.get("transferred_from_other")
    rationale = conversation_data.get("rationale")
    confidence = conversation_data.get("confidence")

    rows = pd.concat(
        [
            rows,
            pd.DataFrame(
                {
                    "interaction_type": [interaction_type],
                    "task_match": [task_match],
                    "label": [label],
                    "lt1_reason": [lt1_reason],
                    "transferred_from": [transferred_from],
                    "transferred_from_other": [transferred_from_other],
                    "rationale": [rationale],
                    "confidence": [confidence],
                }
            ),
        ],
        ignore_index=True,
    )
final_df = pd.concat([sample_50_df.reset_index(drop=True), rows], axis=1)
final_df.drop(columns=["labor_transfer"], inplace=True)


In [ ]:
labels_distribution = final_df["label"].value_counts(normalize=True) * 100
print(f"Label distribution:\n{labels_distribution}")

In [ ]:
task_match_distribution = final_df["task_match"].value_counts(normalize=True) * 100
print(f"Task match distribution:\n{task_match_distribution}")